# SeeSaw — Notebook 2: Gemma 3 1B LoRA Fine-Tuning

Fine-tunes `google/gemma-3-1b-it` on SeeSaw story data using LoRA.

**Input:** `gs://seesaw-models/training-data/seesaw_beats_train.jsonl`  
**Output:** `gs://seesaw-models/checkpoints/seesaw-gemma3-v1/`

**Runtime:** Colab T4 GPU (free tier), ~3 hours

**Prerequisites:**
- Accept Gemma 3 license at huggingface.co/google/gemma-3-1b-it
- HuggingFace token with read access

See `docs/FINE_TUNING.md` for hyperparameter rationale.

In [1]:
# Step 1: Install dependencies
!pip install -q transformers peft datasets accelerate trl google-cloud-storage bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 52.8 MB/s eta 0:00:00


In [2]:
# Step 2: Authenticate — GCP + HuggingFace
from google.colab import auth
auth.authenticate_user()

import subprocess
subprocess.run(['gcloud', 'config', 'set', 'project', 'seesaw-3e396'], check=True)

from huggingface_hub import login
# Paste your HF token when prompted (needs read access + gemma-3-1b-it license accepted)
login()
print('Auth complete')

Auth complete


In [3]:
# Step 3: Load model + tokenizer
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

MODEL_NAME  = 'google/gemma-3-1b-it'
OUTPUT_DIR  = '/tmp/seesaw-gemma3-checkpoint'
GCS_OUTPUT  = 'gs://seesaw-models/checkpoints/seesaw-gemma3-v1'

# 4-bit quantisation for T4 GPU (15 GB VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print(f'Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters')

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Model loaded: 651,005,056 parameters


In [4]:
# Step 4: Apply LoRA
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,981,888 || all params: 1,002,867,840 || trainable%: 0.2973


In [5]:
# Step 5: Load dataset from GCS
dataset = load_dataset('json', data_files={'train': 'gs://seesaw-models/training-data/seesaw_beats_train.jsonl'})
dataset = dataset['train'].train_test_split(test_size=0.1, seed=42)
print(f"Train: {len(dataset['train'])}, Eval: {len(dataset['test'])}")

Generating train split: 0 examples [00:00, ? examples/s]

Train: 7200, Eval: 800


In [24]:
# Step 6: Train
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=100,
    max_grad_norm=1.0,
    fp16=False,   # Gemma 3 uses BFloat16 — FP16 AMP conflicts on T4
    bf16=False,   # T4 does not support BF16 natively
    logging_steps=50,
    save_strategy='epoch',
    eval_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
print(f'Training complete. Final eval_loss: {trainer.state.best_metric:.4f}')

Epoch,Training Loss,Validation Loss
1,0.512594,0.511471
2,0.476859,0.496011
3,0.468724,0.494478


Training complete. Final eval_loss: 0.4945


In [25]:
# Step 7: Upload checkpoint to GCS
import subprocess
result = subprocess.run(
    ['gsutil', '-m', 'cp', '-r', OUTPUT_DIR + '/*', GCS_OUTPUT],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
print(f'Checkpoint uploaded to {GCS_OUTPUT}')

Copying file:///tmp/seesaw-gemma3-checkpoint/checkpoint-900/trainer_state.json [Content-Type=application/json]...
/ [0/40 files][    0.0 B/190.6 MiB]   0% Done                                   
Copying file:///tmp/seesaw-gemma3-checkpoint/checkpoint-900/scheduler.pt [Content-Type=application/vnd.snesdev-page-table]...
/ [0/40 files][    0.0 B/190.6 MiB]   0% Done                                   
Copying file:///tmp/seesaw-gemma3-checkpoint/checkpoint-900/adapter_model.safetensors [Content-Type=application/octet-stream]...
/ [0/40 files][    0.0 B/190.6 MiB]   0% Done                                   
Copying file:///tmp/seesaw-gemma3-checkpoint/checkpoint-900/chat_template.jinja [Content-Type=application/octet-stream]...
/ [0/40 files][    0.0 B/190.6 MiB]   0% Done                                   
Copying file:///tmp/seesaw-gemma3-checkpoint/checkpoint-900/rng_state.pth [Content-Type=application/octet-stream]...
/ [0/40 files][    0.0 B/190.6 MiB]   0% Done                      